In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys, importlib, json, shutil
import numpy as np
import pandas as pd
sys.path.insert(0, '/content/drive/MyDrive/RadarGestureEdgeAI')
import radar_utils as ru
importlib.reload(ru)

DEMO = ru.PROJECT / 'demo'
(DEMO / 'models').mkdir(parents=True, exist_ok=True)
Xc, y, meta = ru.load_compact()

# 5 random recordings per (participant, gesture), duplicates excluded
rng = np.random.default_rng(0)
keep = []
for _, grp in meta[~meta['is_duplicate']].groupby(['participant_id', 'label']):
    keep += list(rng.choice(grp.index.to_numpy(), size=min(5, len(grp)), replace=False))
keep = np.sort(np.array(keep))
np.savez_compressed(DEMO / 'demo_data.npz', X=Xc[keep].astype(np.float16),
                    y=y[keep], subject=meta['participant_id'].to_numpy()[keep])

# LOSO INT8 model + input normalisation for every participant (model never saw that person)
norm = {}
for fold in ru.loso_folds(meta):
    k, s = fold['fold'], int(fold['test_subject'])
    _, mu, sd = ru.load_fold_model('final_M', k)
    norm[str(s)] = {'mean': mu.tolist(), 'std': sd.tolist()}
    shutil.copy(ru.PROJECT / 'models' / f'final_M_fold{k}_int8.tflite',
                DEMO / 'models' / f'subject{s}_int8.tflite')
json.dump(norm, open(DEMO / 'norm.json', 'w'), indent=1)

print("Demo recordings:", len(keep))
for f in sorted(DEMO.rglob('*')):
    if f.is_file():
        print(f"  {f.relative_to(DEMO)}  {f.stat().st_size / 1024:.0f} KB")

Demo recordings: 470
  demo_data.npz  7108 KB
  models/subject1_int8.tflite  30 KB
  models/subject2_int8.tflite  30 KB
  models/subject3_int8.tflite  30 KB
  models/subject4_int8.tflite  30 KB
  models/subject5_int8.tflite  30 KB
  models/subject6_int8.tflite  30 KB
  models/subject7_int8.tflite  30 KB
  models/subject8_int8.tflite  30 KB
  norm.json  1 KB


In [3]:
%%writefile /content/drive/MyDrive/RadarGestureEdgeAI/demo/app.py
"""Privacy-Safe Radar Gesture Recognition using Edge-AI — interactive demo.

Runs the real deployed model: full-INT8 TFLite TinyRadarCNN (30 KB).
Each participant is classified by the leave-one-subject-out model that never saw them.
"""
import json
import time
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr

try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter

HERE = Path(__file__).parent
DATA = np.load(HERE / 'demo_data.npz')
X, Y, SUBJ = DATA['X'], DATA['y'], DATA['subject']
NORM = json.load(open(HERE / 'norm.json'))

# Gesture names (G1..G12). VERIFY against the dataset paper: Ahmed et al., Scientific Data 2021.
GESTURES = {
    1: 'Left→Right swipe',      2: 'Right→Left swipe',
    3: 'Up→Down swipe',         4: 'Down→Up swipe',
    5: 'Diagonal LR, up→down',  6: 'Diagonal LR, down→up',
    7: 'Diagonal RL, up→down',  8: 'Diagonal RL, down→up',
    9: 'Clockwise rotation',   10: 'Anticlockwise rotation',
    11: 'Inward push',         12: 'Empty (no gesture)',
}
name = lambda c: f"G{c + 1} · {GESTURES[c + 1]}"          # c = class index 0..11
SUBJ_CHOICES = ['Any'] + [f'Participant {s}' for s in sorted(set(SUBJ.tolist()))]
GEST_CHOICES = ['Any'] + [name(c) for c in range(12)]
RAW_BYTES = 90 * 189 * 3 * 4                               # one raw recording, float32
rng = np.random.default_rng()
_models = {}


def get_model(s):
    if s not in _models:
        it = Interpreter(model_path=str(HERE / 'models' / f'subject{s}_int8.tflite'))
        it.allocate_tensors()
        _models[s] = it
    return _models[s]


def classify(x, s):
    """Same pipeline as on the MCU: per-recording scaling -> z-score -> INT8 -> model."""
    it = get_model(s)
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    x = x.astype(np.float32)
    x = x / (x.std() + 1e-8)
    x = (x - np.array(NORM[str(s)]['mean'], np.float32)) / np.array(NORM[str(s)]['std'], np.float32)
    sc, zp = inp['quantization']
    q = np.clip(np.round(x / sc + zp), -128, 127).astype(np.int8)[None]
    t0 = time.perf_counter()
    it.set_tensor(inp['index'], q)
    it.invoke()
    ms = (time.perf_counter() - t0) * 1000
    o = it.get_tensor(out['index'])[0].astype(np.float32)
    osc, ozp = out['quantization']
    logits = (o - ozp) * osc
    p = np.exp(logits - logits.max())
    return p / p.sum(), ms


def plot_input(x):
    plt.close('all')
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
    for i, title in enumerate(['Left radar', 'Top radar', 'Right radar']):
        axes[i].imshow(x[:, :, i].astype(np.float32).T, aspect='auto', origin='lower', cmap='magma')
        axes[i].set_title(title)
        axes[i].set_xlabel('Time (frames)')
    axes[0].set_ylabel('Range bin')
    fig.tight_layout()
    return fig


def run(subject_choice, gesture_choice):
    mask = np.ones(len(Y), bool)
    if subject_choice != 'Any':
        mask &= SUBJ == int(subject_choice.split()[-1])
    if gesture_choice != 'Any':
        mask &= Y == GEST_CHOICES.index(gesture_choice) - 1
    idx = np.flatnonzero(mask)
    if len(idx) == 0:
        return None, {}, ("**No recording for this combination.** Participant 2 has no G11/G12 "
                          "recordings: they were byte-identical duplicates and were removed in our data audit.")
    i = int(rng.choice(idx))
    x, true, s = X[i], int(Y[i]), int(SUBJ[i])
    p, ms = classify(x, s)
    pred = int(p.argmax())
    ok = pred == true
    md = f"""### {'✅ Correct' if ok else '❌ Wrong'}
- **True gesture:** {name(true)}
- **Predicted:** {name(pred)} ({p[pred] * 100:.1f}% confidence)
- **Participant {s}:** this model has **never seen this person** (leave-one-subject-out)

| On-device facts | |
|---|---|
| Model | TinyRadarCNN, full INT8, **30 KB** |
| Inference here (cloud CPU) | {ms:.2f} ms |
| Measured on STM32H743 (Cortex-M7, 480 MHz) | 22.9 ms |
| Measured on STM32F469 (Cortex-M4, 180 MHz) | 106.8 ms |
| Data leaving the device | **1 byte** (gesture ID) instead of {RAW_BYTES / 1024:.0f} KB of raw radar |
"""
    return plot_input(x), {name(c): float(p[c]) for c in range(12)}, md


HEADER = """# 📡 Privacy-Safe Radar Gesture Recognition using Edge-AI
Three UWB radars (Left / Top / Right) record a hand gesture. A **30 KB INT8 CNN** that fits on a
microcontroller recognises it **on the device**, so only the gesture label ever leaves the sensor.
*LOSO macro-F1: 89.3% (INT8) on 8 participants · 12 gestures (UWB-Gestures dataset).*
"""
FOOTER = """---
**Why on-device = privacy:** an attacker identifies the user from raw radar with **98.7%** accuracy,
but from the label-only output just **13.4%** (chance = 12.8%).
"""

with gr.Blocks(title='Privacy-Safe Radar Gesture Recognition') as demo:
    gr.Markdown(HEADER)
    with gr.Row():
        subj_dd = gr.Dropdown(SUBJ_CHOICES, value='Any', label='Participant')
        gest_dd = gr.Dropdown(GEST_CHOICES, value='Any', label='Gesture')
        btn = gr.Button('🎲 Pick a recording & classify', variant='primary')
    plot = gr.Plot(label='Radar input seen by the model (3 radars × 90 frames × 32 range bins)')
    with gr.Row():
        probs = gr.Label(num_top_classes=3, label='Edge-model output')
        info = gr.Markdown()
    btn.click(run, [subj_dd, gest_dd], [plot, probs, info])
    gr.Markdown(FOOTER)

if __name__ == '__main__':
    demo.launch()

Writing /content/drive/MyDrive/RadarGestureEdgeAI/demo/app.py


In [4]:
%%writefile /content/drive/MyDrive/RadarGestureEdgeAI/demo/requirements.txt
ai-edge-litert
numpy
matplotlib

Writing /content/drive/MyDrive/RadarGestureEdgeAI/demo/requirements.txt


In [5]:
!pip -q install gradio
import os, sys
os.chdir('/content/drive/MyDrive/RadarGestureEdgeAI/demo')
sys.path.insert(0, '.')
import app
app.demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d20008e986880ba393.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
p = '/content/drive/MyDrive/RadarGestureEdgeAI/demo/app.py'
code = open(p).read()
code = code.replace("fig.tight_layout()\n    return fig",
                    "fig.tight_layout(rect=(0, 0, 1, 0.88))\n    return fig")
code = code.replace(
    "        it = Interpreter(model_path=str(HERE / 'models' / f'subject{s}_int8.tflite'))",
    "        f = HERE / 'models' / f'subject{s}_int8.tflite'\n"
    "        f = f if f.exists() else HERE / f.name      # works with or without models/ folder\n"
    "        it = Interpreter(model_path=str(f))")
open(p, 'w').write(code)
print("plot fix:", code.count('rect=(0, 0, 1, 0.88)'), "| model-path fix:", code.count('f.exists()'))

plot fix: 1 | model-path fix: 1
